<div dir="rtl" align="right">

# NB22E — آزمایشگاه بازیابی، v1

فایل را با همین نام `22e_retrieval_experiment_lab_v1.ipynb` در `notebooks/phase2/` قرار بده؛ **Restart Kernel → Run All**. تنظیم Root اختیاری `RAG_FINANCE_ROOT` است.

**ساختار توافق‌شده:** دو سل مشترک؛ هر مسیر یک سل کامل مستقل؛ یک سل مقایسهٔ نهایی. سل مسیر فقط به Bootstrap مشترک وابسته است و به نتیجهٔ مسیر دیگر متکی نیست. کد inference، ذخیره و ارزیابی برای جلوگیری از اختلاف پیاده‌سازی در موتور مشترک تعریف شده است.

**پنج مسیر:** مرجع NB22D؛ مدل ONNX روی ۲۰۰، ۴۰۰ و ۸۰۰ کاندیدای واقعی `dual800`؛ و ترکیب رتبهٔ مدل۴۰۰ با رتبهٔ Pool۴۰۰. در v1 همهٔ مسیرهای مدل از مدل محلی پین‌شدهٔ قبلی استفاده می‌کنند؛ مدل تازه‌ای دانلود نمی‌شود. این مقایسهٔ مدل‌های مختلف نیست؛ نسخه‌های بعدی فقط با مدل و قرارداد واقعاً موجود اضافه می‌شوند.

**نمونهٔ مشترک:** ۱۲ سؤال Development؛ چهار سؤال کتاب، چهار صوت و چهار وب؛ مؤلفهٔ منبع Gold متمایز، انتخاب قطعی بدون نگاه به نتیجه. این یک پایلوت کوچک برای حذف مسیر نامناسب و سنجش کارایی است، نه آزمون نهایی یا تضمین برندهٔ کل Development. نمونه را پس از دیدن نتیجه عوض نکن.

**اجرا محلی است:** API، هزینهٔ سرویس، دانلود و Test تازه صفر؛ ولی برخلاف NB22D، **inference تازهٔ CPU** روی جفت‌های فاقد امتیاز انجام می‌شود. حداکثر ۱۲۰۰۰ جفت فاقد کش و یک ساعت زمان کل، با ذخیره پس از هر batch چهارتایی؛ عبور از حد، مسیر را Held می‌کند. اگر مدل/نسخه/کش لازم موجود نباشد، دلیل دقیق چاپ و مسیر مدل مسدود می‌شود؛ مرجع مستقل همچنان قابل گزارش است. زمان واقعی روی دستگاه کاربر معلوم می‌شود.

قبل از ترکیب امتیازهای قدیمی و جدید، چهار جفت قبلی فقط برای کنترل تطابق runtime بازسنجی می‌شوند؛ شمارش آن‌ها از جفت‌های فاقد کش جداست. امتیازهای موفق NB22B و baseline50 با قرارداد و متن دقیق مصرف می‌شوند؛ کش جدید سؤال–قطعه بین مسیرها مشترک است و ورودی مدل/Revision/متن را کنترل می‌کند. هر مسیر گزارش و رتبهٔ جدا دارد. مسیر۴۰۰فیوژن در صورت کامل بودن امتیازهای۴۰۰ به inference جدید احتیاج ندارد. با اجرای دوباره، امتیازهای ذخیره‌شده مصرف می‌شوند؛ اگر batch در میانهٔ اجرا قطع شود ممکن است همان batch محلیِ ذخیره‌نشده دوباره محاسبه شود، بدون API.

پوشهٔ جدید گزارش `reports/phase2/quality_improvement/NB22E.lab.v1/` و کش جفت‌ها `artifacts/phase2/quality_improvement/NB22E.pair_cache_v1/` است. قرارداد امتیاز از تغییر جدول و سل‌ها مستقل است تا ویرایش گزارش باعث از دست رفتن کش نشود. تنها یک kernel روی این کش اجرا کن؛ نوشتن همزمان چند kernel پشتیبانی نمی‌شود.

جدول نهایی Recall@10 با حد۸۵٪ و nDCG@10 با حد۷۵٪ را همراه ✅ فقط برای عبور از حد چاپ می‌کند. Hit@10 حد مطلق مصوب ندارد. علامت سبز فقط مربوط به همین پایلوت است؛ Release پذیرفته نمی‌شود. مسیر ناقص با سؤال کمتر وارد انتخاب نمی‌شود. زمانِ هر مسیر افزایشی و متأثر از کش مشترک است و سرعت مستقل مدل نام نمی‌گیرد.

هیچ اجرای خودکار روی کل Development یا Test انجام نمی‌شود. بعد از این مقایسه، فقط مسیر امیدوارکننده با بررسی کیفیت، موارد افت و زمان/بودجهٔ واقعی برای آزمایش بزرگ‌تر انتخاب می‌شود. وضعیت‌های باز معنایی/ایمنی، رأی‌های Held قبلی و تعویق اقدام۴۸ محفوظ‌اند.

</div>

<div dir="rtl" align="right">

### NB22E-C001 — Imports

فقط Imports استاندارد.

</div>

In [1]:
# NB22E-C001 — Imports only; standard library, no file/network/model operations.
import collections
import hashlib
import json
import math
import os
import platform
import re
import statistics
import tempfile
import time
from pathlib import Path


<div dir="rtl" align="right">

### NB22E-C002 — Bootstrap، اتصال ورودی و موتور مشترک

بستهٔ مشترک: هویت، کنترل ورودی و کش، انتخاب نمونه و ثبت طرح پیش از inference.

</div>

In [2]:
# NB22E-C002 — Shared settings, exact inputs, engine and frozen pilot.
PINS = {'C056': {'path': 'reports/phase2/release/NB22.C056.retrieval_verification/C056.independent_retrieval_metric_verification.ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22.json', 'file_bytes_sha256': 'ff6d6e136043929e0156e43f6ea75e65bd64f40ab7bcdafef90a4ba40901de22', 'bytes': 1180779, 'write_status': 'CREATED', 'reload_validation': 'PASS'}, 'C_final': {'path': 'reports/phase2/quality_improvement/NB22C.v1/A06.final_report.1797d76fe08136a50ffa846da39858f29838a9bc8a3dc3d1cad120789ccc53ea.json', 'file_bytes_sha256': '1797d76fe08136a50ffa846da39858f29838a9bc8a3dc3d1cad120789ccc53ea', 'bytes': 31539, 'reload_validation': 'PASS'}, 'C_rankings': {'path': 'reports/phase2/quality_improvement/NB22C.v1/A05.retained_development_rankings.5bc216aae839bc80cc2f184ca3a3cd3e4ac33657655c3bffb4fac410e96e9190.jsonl', 'file_bytes_sha256': '5bc216aae839bc80cc2f184ca3a3cd3e4ac33657655c3bffb4fac410e96e9190', 'bytes': 19285045, 'reload_validation': 'PASS'}, 'C_diagnosis': {'path': 'reports/phase2/quality_improvement/NB22C.v1/A02.development_diagnosis.2b86a0709a76b3e5673383d6f95d973991890246cb129faf7fe995e8f28dcdf5.jsonl', 'file_bytes_sha256': '2b86a0709a76b3e5673383d6f95d973991890246cb129faf7fe995e8f28dcdf5', 'bytes': 1610871, 'reload_validation': 'PASS'}, 'D_final': {'path': 'reports/phase2/quality_improvement/NB22D.v1/A05.final_report.4b1626f9669aa7bc8ffa0ada5ddffe8d7350aa8dcf5db9e956bd8c67f86c82de.json', 'file_bytes_sha256': '4b1626f9669aa7bc8ffa0ada5ddffe8d7350aa8dcf5db9e956bd8c67f86c82de', 'bytes': 11977, 'reload_validation': 'PASS'}, 'B_cache_inventory': {'bytes': 760689, 'file_bytes_sha256': '3ebb5c79d71e0ee21ffab7e99a1f32580547a1d1f7a7b0a8a7cb693a294c0e93', 'path': 'reports/phase2/quality_improvement/NB22B.v1/A05.local_score_cache_inventory.3ebb5c79d71e0ee21ffab7e99a1f32580547a1d1f7a7b0a8a7cb693a294c0e93.json', 'reload_validation': 'PASS'}}
MODEL = {'model_id': 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1', 'revision': '1427fd652930e4ba29e8149678df786c240d8825', 'model_sha256': '6c2513767fb63d008a4377bef7a7a3555433d9436342bb53e35a3a72ffc52d4b', 'onnxruntime_version': '1.16.3', 'batch_size': 4, 'maximum_sequence_length': 512, 'implementation_ast_sha256': 'd6713ab6ba672379e122d0712b2a6f4a7a9dd40eedfb437c5262366cd766f4c5'}
ROUTES = [{'name': 'baseline_D', 'mode': 'baseline'}, {'name': 'onnx200', 'mode': 'rerank', 'depth': 200}, {'name': 'onnx400', 'mode': 'rerank', 'depth': 400}, {'name': 'onnx800', 'mode': 'rerank', 'depth': 800}, {'name': 'onnx400_fusion', 'mode': 'fusion', 'depth': 400}]

# Shared setup: explicit offline CPU pilot, no model download or paid API.
CONFIG={'version':'NB22E_LAB_V1','seed':'NB22E_v1_SOURCE_PILOT',
 'pilot_per_source_type':4,'maximum_new_pairs_per_run':12000,'maximum_local_seconds_per_run':3600,
 'thresholds':{'recall_at_10':.85,'ndcg_at_10':.75},'rrf_k':10,'reranker_weight':.75}
_start=Path(os.environ.get('RAG_FINANCE_ROOT',str(Path.cwd()))).expanduser().resolve()
ROOT=next((p for p in [_start,*_start.parents] if (p/'notebooks/phase2').is_dir() and
 (p/'data/Road_Maps/RAG_finance_phase2_modular_notebooks_execution_plan.md').is_file()),None)
STATE={'actions':{},'artifacts':[],'memory':{},'errors':[],'inputs':{},'api_calls':0,'model_pairs':0,'network_calls':0}
LAB={'ready':False,'results':{},'issues':[],'started':None,'new_pairs':0,'predict':None,'score_memory':{},'new_inference_seconds':0.0,'parity_pairs':0,'last_progress':0.0}
NOTEBOOK_CODE_HASH=None;SAVED_CODE_VALID=False
if ROOT is not None:
 p=ROOT/'notebooks/phase2/22e_retrieval_experiment_lab_v1.ipynb'
 if p.is_file():
  n=json.loads(p.read_bytes());codes=[''.join(c['source']) if isinstance(c['source'],list) else c['source'] for c in n['cells'] if c['cell_type']=='code']
  NOTEBOOK_CODE_HASH=hashlib.sha256(json.dumps(codes,ensure_ascii=False,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()
  SAVED_CODE_VALID=NOTEBOOK_CODE_HASH==n.get('metadata',{}).get('nb22e',{}).get('code_source_sha256')

# NB22E — shared mechanics — Shared mechanics; no historical notebook is executed.
class InputGap(RuntimeError):
    pass

def require(ok, reason):
    if not ok:
        raise InputGap(reason)

def canonical(x):
    return json.dumps(x, ensure_ascii=False, sort_keys=True,
                      separators=(',', ':'), allow_nan=False).encode('utf-8')

def digest(raw):
    return hashlib.sha256(raw).hexdigest()

def decode(raw):
    def pairs(items):
        out = {}
        for k, v in items:
            if k in out:
                raise InputGap('DUPLICATE_JSON_KEY:' + k)
            out[k] = v
        return out
    def reject(x):
        raise InputGap('NONFINITE_JSON:' + x)
    return json.loads(raw, object_pairs_hook=pairs, parse_constant=reject)

def path_for(relative):
    require(ROOT is not None, 'PROJECT_ROOT_REQUIRED:run_inside_repository_or_set_RAG_FINANCE_ROOT')
    p = Path(relative)
    require(not p.is_absolute() and '..' not in p.parts, 'PROJECT_RELATIVE_PATH_REQUIRED')
    target = (ROOT / p).resolve()
    require(target.is_relative_to(ROOT), 'PATH_OUTSIDE_PROJECT')
    return target

def binding_hash(binding):
    values = {binding[k] for k in ('file_bytes_sha256', 'sha256') if k in binding}
    require(len(values) == 1, 'UNAMBIGUOUS_BYTE_HASH_REQUIRED')
    h = next(iter(values))
    require(isinstance(h, str) and re.fullmatch('[a-f0-9]{64}', h), 'INVALID_HASH')
    return h

def checked_bytes(binding):
    p = path_for(binding['path'])
    require(p.is_file(), 'REQUIRED_FILE_NOT_AVAILABLE:' + binding['path'])
    before = p.stat()
    raw = p.read_bytes()
    after = p.stat()
    require((before.st_size, before.st_mtime_ns, before.st_ino) ==
            (after.st_size, after.st_mtime_ns, after.st_ino), 'SOURCE_CHANGED_DURING_READ')
    require(digest(raw) == binding_hash(binding), 'INPUT_BYTE_HASH_MISMATCH:' + binding['path'])
    STATE['inputs'][binding['path']] = dict(binding)
    return raw

def read_json(binding):
    return decode(checked_bytes(binding))

def read_rows(binding):
    raw = checked_bytes(binding)
    require(all(line.strip() for line in raw.splitlines()), 'BLANK_JSONL_ROW')
    return [decode(line) for line in raw.splitlines()]

def keyed(rows, field):
    out = {}
    for row in rows:
        require(isinstance(row, dict) and isinstance(row.get(field), str) and
                row[field] not in out, 'MISSING_OR_DUPLICATE_ID:' + field)
        out[row[field]] = row
    return out

def save(stem, value, rows=False):
    folder = path_for('reports/phase2/quality_improvement/NB22E.lab.v1')
    folder.mkdir(parents=True, exist_ok=True)
    raw = b''.join(canonical(r) + b'\n' for r in value) if rows else canonical(value)
    suffix = 'jsonl' if rows else 'json'
    p = folder / (stem + '.' + digest(raw) + '.' + suffix)
    if not p.exists():
        fd, name = tempfile.mkstemp(dir=folder, prefix='.nb22e-')
        try:
            with os.fdopen(fd, 'wb') as f:
                f.write(raw); f.flush(); os.fsync(f.fileno())
            try:
                os.link(name, p)
            except FileExistsError:
                pass
        finally:
            Path(name).unlink(missing_ok=True)
    actual = p.read_bytes()
    require(actual == raw, 'OUTPUT_COLLISION_OR_RELOAD_FAILURE')
    if rows:
        require([decode(x) for x in actual.splitlines()] == value, 'OUTPUT_ROW_RELOAD_FAILURE')
    else:
        require(decode(actual) == value, 'OUTPUT_RELOAD_FAILURE')
    receipt = {'path': p.relative_to(ROOT).as_posix(), 'file_bytes_sha256': digest(raw),
               'bytes': len(raw), 'reload_validation': 'PASS'}
    STATE['artifacts'].append(receipt)
    return receipt

def metrics(order, gold):
    require(gold and len(order) == len(set(order)), 'INVALID_METRIC_POPULATION')
    flags = [int(x in gold) for x in order[:50]]
    out = {}
    for k in (5, 10):
        hits = sum(flags[:k])
        idcg = math.fsum(1 / math.log2(i + 2) for i in range(min(k, len(gold))))
        out.update({f'hit_at_{k}': float(hits > 0), f'precision_at_{k}': hits / k,
                    f'recall_at_{k}': hits / len(gold),
                    f'ndcg_at_{k}': math.fsum(v / math.log2(i + 2) for i, v in enumerate(flags[:k])) / idcg})
    first = next((i + 1 for i, v in enumerate(flags) if v), None)
    out['reciprocal_rank'] = 1 / first if first else 0.0
    return out

def mean_metric(values):
    rows = list(values)
    require(bool(rows), 'EMPTY_METRIC_COHORT')
    return {k: math.fsum(r[k] for r in rows) / len(rows) for k in sorted(rows[0])}

def rank_ids(row, count):
    c = row['candidates']
    require(len(c) == count and [r['rank'] for r in c] == list(range(1, count + 1)), 'RANK_SEQUENCE_OR_DEPTH_CONFLICT')
    ids = [r['chunk_id'] for r in c]
    require(len(ids) == len(set(ids)), 'DUPLICATE_RANKED_CHUNK')
    return ids

def run_action(number, fn, dependencies=()):
    blocked = [d for d in dependencies if STATE['actions'].get(d, {}).get('status') != 'COMPLETE_IN_SCOPE']
    if blocked:
        out = {'status': 'BLOCKED_BY_DEPENDENCY', 'dependencies': blocked}
    else:
        try:
            out = fn()
            out['status'] = 'COMPLETE_IN_SCOPE'
        except InputGap as error:
            out = {'status': 'BLOCKED_INPUT_OR_CONTRACT', 'reason': str(error)}
        except Exception as error:
            # Real implementation errors remain distinguishable from missing input.
            out = {'status': 'FAILED_IMPLEMENTATION', 'error_type': type(error).__name__}
            STATE['errors'].append({'action': number, 'error_type': type(error).__name__})
    STATE['actions'][number] = out
    print(json.dumps({'action': number, **out, 'new_api_calls': 0}, ensure_ascii=False, indent=2))
    return out

def components(dev):
    parent = {q: q for q in dev['ids']}
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    seen = {}
    for q in dev['ids']:
        for cid in dev['gold'][q]:
            sid = dev['chunks'][cid].get('source_id')
            require(isinstance(sid, str) and sid.strip(), 'GOLD_SOURCE_PROVENANCE_REQUIRED')
            if sid in seen:
                a, b = find(q), find(seen[sid])
                parent[max(a, b)] = min(a, b)
            else:
                seen[sid] = q
    return {q: find(q) for q in dev['ids']}

def partition(component_map):
    labels = {c: ('validation' if int(digest((CONFIG['seed'] + c).encode())[:16], 16) % 100
                    < CONFIG['validation_percent'] else 'tuning') for c in set(component_map.values())}
    out = {name: sorted(q for q, c in component_map.items() if labels[c] == name)
           for name in ('tuning', 'validation')}
    for name, ids in out.items():
        require(len({component_map[q] for q in ids}) >= CONFIG['minimum_components_per_partition'],
                'INSUFFICIENT_SOURCE_COMPONENTS:' + name)
    return out

def eligible(candidate, reference):
    return all(candidate[k] >= reference[k] - 1e-12 for k in ('recall_at_10', 'hit_at_10')) and candidate['ndcg_at_10'] > reference['ndcg_at_10'] + 1e-12


def load_local_predictor():
 if LAB['predict'] is not None:return LAB['predict']
 print(json.dumps({'stage':'INITIALIZE_PINNED_LOCAL_ONNX','downloads':False}),flush=True)
 try:
  import numpy as np
  import onnxruntime as ort
  from transformers import AutoTokenizer
 except ImportError as e:raise InputGap('LOCAL_RUNTIME_PACKAGE_UNAVAILABLE:'+str(e.name)) from e
 require(ort.__version__==MODEL['onnxruntime_version'],'PINNED_ONNXRUNTIME_VERSION_REQUIRED')
 cache_root=Path(os.environ.get('HF_HUB_CACHE') or str(Path(os.environ.get('HF_HOME',str(Path.home()/'.cache/huggingface')))/'hub'))
 snapshot=cache_root/('models--'+MODEL['model_id'].replace('/','--'))/'snapshots'/MODEL['revision']
 model=snapshot/'onnx/model_quint8_avx2.onnx'
 require(model.is_file(),'PINNED_LOCAL_ONNX_NOT_CACHED_NO_DOWNLOAD')
 require(digest(model.read_bytes())==MODEL['model_sha256'],'MODEL_BYTES_IDENTITY_CONFLICT')
 tokenizer=AutoTokenizer.from_pretrained(str(snapshot),local_files_only=True,trust_remote_code=False)
 opts=ort.SessionOptions();opts.intra_op_num_threads=4;opts.inter_op_num_threads=1
 opts.execution_mode=ort.ExecutionMode.ORT_SEQUENTIAL;opts.graph_optimization_level=ort.GraphOptimizationLevel.ORT_ENABLE_ALL
 session=ort.InferenceSession(str(model),sess_options=opts,providers=['CPUExecutionProvider'])
 names={x.name for x in session.get_inputs()}
 require({'input_ids','attention_mask'}.issubset(names),'ONNX_INPUT_CONTRACT_CONFLICT')
 def predict(q,ids):
  require(0<len(ids)<=4,'BATCH_SIZE_CONFLICT')
  encoded=tokenizer([LAB['questions'][q]]*len(ids),[LAB['dev']['chunks'][c]['content'] for c in ids],
    padding=True,truncation=True,max_length=512,return_tensors='np')
  feed={name:np.asarray(encoded[name],dtype=np.int64) for name in names if name in encoded}
  require(set(feed)==names,'TOKENIZER_MODEL_INPUT_CONFLICT')
  values=np.asarray(session.run(None,feed)[0],dtype=np.float64).reshape(-1)
  require(values.shape==(len(ids),) and bool(np.isfinite(values).all()),'INVALID_MODEL_SCORES')
  return [float(v) for v in values]
 # Recheck current CPU runtime parity before mixing cached and new model scores.
 q=LAB['pilot'][0];sample=list(LAB['legacy'][q])[:4]
 observed=predict(q,sample);LAB['parity_pairs']+=len(sample);delta=max(abs(v-LAB['legacy'][q][cid]) for v,cid in zip(observed,sample))
 require(delta<=1e-4,'LOCAL_SCORE_PARITY_FAILURE_NO_NEW_SCORE_MIXING')
 LAB['guard']=save('runtime_parity',{'model_contract_sha256':LAB['model_contract']['sha256'],
  'model_sha256':MODEL['model_sha256'],'onnxruntime_version':ort.__version__,'sample_max_delta':delta,
  'sample_query':q,'sample_chunks':sample,'scope':'FIRST4_SAMPLE_ONLY_NOT_FULL_NUMERICAL_EQUIVALENCE'})
 LAB['predict']=predict
 return predict

def score_cache(q):
 identity={'schema_version':'NB22E_PAIR_CACHE_V1','model_contract_sha256':LAB['model_contract']['sha256'],
  'candidate_id':q,'question_sha256':digest(LAB['questions'][q].encode())}
 folder=path_for('artifacts/phase2/quality_improvement/NB22E.pair_cache_v1')
 folder.mkdir(parents=True,exist_ok=True);p=folder/(digest(canonical(identity))+'.json')
 if p.is_file():
  before=p.stat();raw=p.read_bytes();after=p.stat();require((before.st_size,before.st_mtime_ns)==(after.st_size,after.st_mtime_ns),'PAIR_CACHE_CHANGED_DURING_READ')
  record=decode(raw);require(record['identity']==identity and record['pairs_sha256']==digest(canonical(record['pairs'])),'PAIR_CACHE_IDENTITY_OR_CHECKSUM_CONFLICT')
  guard=read_json(record['runtime_guard'])
  require(guard['model_contract_sha256']==identity['model_contract_sha256'] and guard['model_sha256']==MODEL['model_sha256'] and 0<=guard['sample_max_delta']<=1e-4,'PAIR_CACHE_RUNTIME_GUARD_CONFLICT')
  for cid,item in record['pairs'].items():
   require(cid in LAB['dev']['chunks'] and item['content_sha256']==digest(LAB['dev']['chunks'][cid]['content'].encode()) and type(item['score']) in (int,float) and math.isfinite(item['score']),'PAIR_CACHE_TEXT_OR_VALUE_CONFLICT')
 else:record={'identity':identity,'pairs':{}}
 return p,record

def persist_pairs(path,record):
 record['pairs_sha256']=digest(canonical(record['pairs']));raw=canonical(record)
 fd,name=tempfile.mkstemp(dir=path.parent,prefix='.nb22e-score-')
 try:
  with os.fdopen(fd,'wb') as f:f.write(raw);f.flush();os.fsync(f.fileno())
  os.replace(name,path)
 finally:Path(name).unlink(missing_ok=True)
 require(path.read_bytes()==raw and decode(path.read_bytes())==record,'PAIR_SCORE_CHECKPOINT_RELOAD_FAILURE')

def score_order(q,order):
 path,record=score_cache(q)
 scores=dict(LAB['legacy'][q]);scores.update({c:x['score'] for c,x in record['pairs'].items()})
 missing=[c for c in order if c not in scores]
 for start in range(0,len(missing),4):
  batch=missing[start:start+4]
  require(LAB['new_pairs']+len(batch)<=CONFIG['maximum_new_pairs_per_run'],'NEW_PAIR_BUDGET_REACHED_RESUME_CACHE_SAFE')
  require(time.monotonic()-LAB['started']<CONFIG['maximum_local_seconds_per_run'],'LOCAL_TIME_CAP_REACHED_RESUME_CACHE_SAFE')
  predict=load_local_predictor();t=time.monotonic();values=predict(q,batch)
  LAB['new_inference_seconds']+=time.monotonic()-t
  require(len(values)==len(batch) and all(type(x) in (int,float) and math.isfinite(x) for x in values),'INVALID_NEW_PAIR_VALUES')
  for c,v in zip(batch,values):
   record['pairs'][c]={'score':v,'content_sha256':digest(LAB['dev']['chunks'][c]['content'].encode())};scores[c]=v
  record['runtime_guard']=LAB['guard'];persist_pairs(path,record)
  LAB['new_pairs']+=len(batch);STATE['model_pairs']+=len(batch)
  if time.monotonic()-LAB['last_progress']>=20:
   print(json.dumps({'stage':'LOCAL_PAIR_SCORING','candidate_id':q,'cached_or_legacy_pairs_for_query':len(scores),
    'required_for_route':len(order),'new_pairs_this_run':LAB['new_pairs'],'parity_check_pairs':LAB['parity_pairs'],
    'elapsed_seconds':round(time.monotonic()-LAB['started'],1)}),flush=True);LAB['last_progress']=time.monotonic()
 return scores

def run_route(spec):
 name=spec['name'];started=time.monotonic();before_pairs=LAB['new_pairs'];before_inference=LAB['new_inference_seconds']
 if not LAB['ready']:
  result={'status':'BLOCKED_SETUP','issues':LAB['issues'],'specification':spec}
 else:
  if LAB['started'] is None:LAB['started']=time.monotonic()
  values={};orders={};pool_ceiling={};held=[]
  for q in LAB['pilot']:
   try:
    if spec['mode']=='baseline':
     order=LAB['incumbent'][q];pool=order
    else:
     pool=LAB['pools'][q][:spec['depth']];scores=score_order(q,pool)
     order=sorted(pool,key=lambda c:(-scores[c],c))
     if spec['mode']=='fusion':
      rr={c:i+1 for i,c in enumerate(order)};pr={c:i+1 for i,c in enumerate(pool)};w=CONFIG['reranker_weight'];k=CONFIG['rrf_k']
      order=sorted(pool,key=lambda c:(-(w/(k+rr[c])+(1-w)/(k+pr[c])),c))
    values[q]=metrics(order,LAB['dev']['gold'][q]);orders[q]=order
    pool_ceiling[q]=min(10,len(set(pool)&LAB['dev']['gold'][q]))/len(LAB['dev']['gold'][q])
   except InputGap as e:held.append({'candidate_id':q,'reason':str(e)})
   except Exception as e:held.append({'candidate_id':q,'reason':'RUNTIME_FAILURE:'+type(e).__name__})
   print(json.dumps({'route':name,'completed':len(values),'held':len(held),'total':len(LAB['pilot']),
    'new_pairs_this_route':LAB['new_pairs']-before_pairs,'elapsed_seconds':round(time.monotonic()-started,1)},ensure_ascii=False),flush=True)
   if held:break  # shared runtime/limit error; preserve partial route and continue independent route cells.
  complete=set(values)==set(LAB['pilot'])
  artifact=save('route_'+name,{'specification':spec,'plan':LAB['plan'],'metrics_by_query':values,'orders':orders,
   'pool_ceiling_by_query':pool_ceiling,'held':held,'complete':complete})
  result={'status':'COMPLETE' if complete else 'PARTIAL_HELD_NOT_ELIGIBLE','specification':spec,
   'query_count':len(values),'expected_queries':len(LAB['pilot']),'metrics':mean_metric(values.values()) if complete else None,
   'pool_ceiling':statistics.fmean(pool_ceiling.values()) if complete else None,
   'new_pairs':LAB['new_pairs']-before_pairs,'route_wall_seconds':time.monotonic()-started,
   'inference_seconds_new_pairs':LAB['new_inference_seconds']-before_inference,
   'timing_scope':'INCREMENTAL_SHARED_CACHE_NOT_FAIR_STANDALONE_MODEL_LATENCY',
   'held':held,'artifact':artifact,'values':values}
 LAB['results'][name]=result
 print(json.dumps({'route':name,**{k:v for k,v in result.items() if k!='values'}},ensure_ascii=False,indent=2))
 return result

def prepare_lab():
 require(SAVED_CODE_VALID,'SAVED_NOTEBOOK_CODE_IDENTITY_REQUIRED')
 verified=read_json(PINS['C056']);previous=read_json(PINS['D_final'])
 require(verified['independent_recomputation_complete'] and previous['pool_ready_for_bounded_reranker_pilot'],'VERIFIED_EXPANSION_REQUIRED')
 bindings=verified['input_file_receipts'];chunks=keyed(read_rows(bindings['chunks']),'chunk_id')
 golden=keyed(read_rows(bindings['golden']),'candidate_id');hybrid=keyed(read_rows(bindings['hybrid']),'candidate_id')
 require(len(chunks)==3488 and len(golden)==3813 and set(golden)==set(hybrid),'POPULATION_CONFLICT')
 loc=collections.defaultdict(set);types=collections.defaultdict(set)
 for cid,c in chunks.items():
  for m in c['member_locators']:
   k=canonical(m['locator']);loc[k].add(cid);types[k].add(c['source_type'])
 gold={}
 for q,row in golden.items():
  g=set()
  for e in row['evidence_locators']:
   k=canonical(e['source_locator']);require(bool(loc[k]) and bool(types[k]&set(row['source_types'])),'GOLD_MAPPING_CONFLICT');g.update(loc[k])
  require(bool(g),'EMPTY_GOLD');gold[q]=g
 md=hashlib.sha256()
 for q in sorted(gold):
  raw=canonical({'candidate_id':q,'relevant_chunk_ids':sorted(gold[q])});md.update(len(raw).to_bytes(8,'big'));md.update(raw)
 require(md.hexdigest()==verified['gold_mapping']['recomputed_sha256'],'LOGICAL_GOLD_CONFLICT')
 ids=sorted(q for q in hybrid if hybrid[q]['split']=='development');require(len(ids)==2664,'DEVELOPMENT_COUNT_CONFLICT')
 dev={'ids':ids,'chunks':chunks,'gold':{q:gold[q] for q in ids}}
 cmap=components(dev);seen=set();pilot=[]
 for source in ('book','transcript','website'):
  candidates=sorted((q for q in ids if golden[q]['source_types']==[source]),key=lambda q:(digest((CONFIG['seed']+q).encode()),q))
  chosen=[]
  for q in candidates:
   if cmap[q] in seen:continue
   chosen.append(q);seen.add(cmap[q])
   if len(chosen)==CONFIG['pilot_per_source_type']:break
  require(len(chosen)==CONFIG['pilot_per_source_type'],'INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:'+source)
  pilot.extend(chosen)
 pool_binding=previous['pool_artifact'];ranking_binding=previous['retained_rankings']
 # Stream large artifacts, validate full hashes, retain only pilot orders.
 def pilot_rows(binding):
  p=path_for(binding['path']);require(p.is_file(),'REQUIRED_FILE_NOT_AVAILABLE:'+binding['path']);before=p.stat();h=hashlib.sha256();out={}
  with p.open('rb') as f:
   for raw in f:
    h.update(raw);row=decode(raw)
    if row.get('candidate_id') in pilot:
     require(row['candidate_id'] not in out,'DUPLICATE_PILOT_ROW');out[row['candidate_id']]=row
  after=p.stat();require((before.st_size,before.st_mtime_ns)==(after.st_size,after.st_mtime_ns) and h.hexdigest()==binding_hash(binding),'PILOT_ARTIFACT_HASH_OR_MUTATION_CONFLICT')
  require(set(out)==set(pilot),'PILOT_COHORT_CONFLICT');STATE['inputs'][binding['path']]=binding;return out
 pool_rows=pilot_rows(pool_binding);rank_rows=pilot_rows(ranking_binding);reranker=pilot_rows(bindings['reranker'])
 pools={};incumbent={};legacy={}
 for q in pilot:
  pool=pool_rows[q]['candidate_chunk_ids'];order=rank_rows[q]['ranked_chunk_ids']
  require(pool_rows[q]['split']==rank_rows[q]['split']==reranker[q]['split']=='development','PILOT_SPLIT_CONFLICT')
  require(len(pool)<=800 and len(pool)==len(set(pool)) and set(rank_ids(hybrid[q],100)).issubset(pool) and set(pool).issubset(chunks),'EXPANDED_POOL_CONFLICT')
  require(set(order).issubset(chunks) and all(abs(v-rank_rows[q]['metrics'][k])<=1e-12 for k,v in metrics(order,gold[q]).items()),'BASELINE_REPLAY_CONFLICT')
  pools[q]=pool;incumbent[q]=order;legacy[q]={c['chunk_id']:float(c['reranker_score']) for c in reranker[q]['candidates']}
 # Exact prior score cache inventory, not a filesystem search or a new model run.
 inventory=read_json(PINS['B_cache_inventory']);contract=inventory['inference_contract'];spec=contract['specification']
 require(digest(canonical(spec))==contract['sha256'],'LEGACY_CONTRACT_CHECKSUM_CONFLICT')
 for key in MODEL:
  require(spec.get(key)==MODEL[key],'LEGACY_MODEL_CONTRACT_CONFLICT:'+key)
 require(spec['baseline_reranker_file_bytes_sha256']==binding_hash(bindings['reranker']),'LEGACY_BASELINE_SCORE_BINDING_CONFLICT')
 require(spec['execution_provider']=='CPUExecutionProvider' and spec['threads']=={'intra_op':4,'inter_op':1},'LEGACY_EXECUTION_PROFILE_CONFLICT')
 found=set()
 for binding in inventory['files']:
  if not binding['path'].startswith('artifacts/phase2/quality_improvement/NB22B.depth100/'):continue
  # Guards and records share a directory; records have the per-query identity.
  record=read_json(binding);identity=record.get('identity',{});q=identity.get('candidate_id')
  if q not in pilot:continue
  require(q not in found and identity['inference_contract_sha256']==contract['sha256'],'LEGACY_QUERY_CONTRACT_CONFLICT')
  question=golden[q]['evaluation_content']['question'].strip()
  require(identity['question_sha256']==digest(question.encode()),'LEGACY_QUESTION_BINDING_CONFLICT')
  require(digest(canonical(record['scores']))==record['score_values_sha256'],'LEGACY_SCORE_CHECKSUM_CONFLICT')
  missing=identity['ordered_missing_chunks'];require(len(missing)==len(record['scores']),'LEGACY_SCORE_COUNT_CONFLICT')
  guard=read_json(record['runtime_validation_receipt'])
  require(guard['inference_contract_sha256']==contract['sha256'] and guard['model_sha256']==MODEL['model_sha256'] and 0<=guard['sample_score_max_delta']<=1e-4,'LEGACY_RUNTIME_GUARD_CONFLICT')
  for item,value in zip(missing,record['scores']):
   cid=item['chunk_id'];require(cid in chunks and item['content_sha256']==digest(chunks[cid]['content'].encode()) and type(value) in (int,float) and math.isfinite(value),'LEGACY_PAIR_BINDING_CONFLICT');legacy[q][cid]=value
  require(set(legacy[q])==set(rank_ids(hybrid[q],100)),'LEGACY100_POOL_CONFLICT');found.add(q)
 require(found==set(pilot),'COMPLETE_PRIOR100_SCORE_CACHE_REQUIRED_NO_RECOMPUTE_FALLBACK')
 LAB.update(ready=True,dev=dev,pilot=pilot,questions={q:golden[q]['evaluation_content']['question'].strip() for q in pilot},
  pools=pools,incumbent=incumbent,legacy=legacy,model_contract=contract,source_types={q:golden[q]['source_types'] for q in pilot})
 plan={'schema_version':CONFIG['version'],'producer_code_sha256':NOTEBOOK_CODE_HASH,'configuration':CONFIG,'pilot':pilot,
  'pilot_selection':'four per source type, deterministic hash order, distinct Gold source components; no outcome-based sampling',
  'source_inputs':STATE['inputs'],'model_contract':contract,
  'routes':ROUTES,'selection':'Exploratory pilot: highest nDCG@10 among full identical-cohort routes with Recall@10 and Hit@10 nondecreasing vs NB22D, strict nDCG gain. Tie route name. No automatic full execution.',
  'scope':'SMALL_DEVELOPMENT_PILOT_ALREADY_EXPOSED_NOT_FINAL_HOLDOUT','test_evaluation':False,'release_acceptance_granted':False,'api_calls':0}
 LAB['plan']=save('A01.frozen_pilot_plan',plan)
 return {'pilot_queries':len(pilot),'source_components':len(pilot),'model_downloads':False,'legacy_queries_reused':len(found),'plan':LAB['plan']}
try:
 print(json.dumps({'setup':'COMPLETE',**prepare_lab()},ensure_ascii=False,indent=2))
except InputGap as e:
 LAB['ready']=False;LAB['issues'].append(str(e));print(json.dumps({'setup':'BLOCKED','reason':str(e)},ensure_ascii=False,indent=2))


{
  "setup": "BLOCKED",
  "reason": "INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:book"
}


<div dir="rtl" align="right">

### NB22E-C003 — مرجع NB22D

اجرای کامل همین مسیر، ذخیره، ارزیابی و گزارش؛ بدون وابستگی به خروجی مسیر دیگر.

</div>

In [3]:
# NB22E-C003 — Independent complete route: baseline_D
SPEC = {'name': 'baseline_D', 'mode': 'baseline'}
RESULT = run_route(SPEC)


{
  "route": "baseline_D",
  "status": "BLOCKED_SETUP",
  "issues": [
    "INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:book"
  ],
  "specification": {
    "name": "baseline_D",
    "mode": "baseline"
  }
}


<div dir="rtl" align="right">

### NB22E-C004 — Reranker روی ۲۰۰ کاندیدا

اجرای کامل همین مسیر، ذخیره، ارزیابی و گزارش؛ بدون وابستگی به خروجی مسیر دیگر.

</div>

In [4]:
# NB22E-C004 — Independent complete route: onnx200
SPEC = {'name': 'onnx200', 'mode': 'rerank', 'depth': 200}
RESULT = run_route(SPEC)


{
  "route": "onnx200",
  "status": "BLOCKED_SETUP",
  "issues": [
    "INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:book"
  ],
  "specification": {
    "name": "onnx200",
    "mode": "rerank",
    "depth": 200
  }
}


<div dir="rtl" align="right">

### NB22E-C005 — Reranker روی ۴۰۰ کاندیدا

اجرای کامل همین مسیر، ذخیره، ارزیابی و گزارش؛ بدون وابستگی به خروجی مسیر دیگر.

</div>

In [5]:
# NB22E-C005 — Independent complete route: onnx400
SPEC = {'name': 'onnx400', 'mode': 'rerank', 'depth': 400}
RESULT = run_route(SPEC)


{
  "route": "onnx400",
  "status": "BLOCKED_SETUP",
  "issues": [
    "INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:book"
  ],
  "specification": {
    "name": "onnx400",
    "mode": "rerank",
    "depth": 400
  }
}


<div dir="rtl" align="right">

### NB22E-C006 — Reranker روی ۸۰۰ کاندیدا

اجرای کامل همین مسیر، ذخیره، ارزیابی و گزارش؛ بدون وابستگی به خروجی مسیر دیگر.

</div>

In [6]:
# NB22E-C006 — Independent complete route: onnx800
SPEC = {'name': 'onnx800', 'mode': 'rerank', 'depth': 800}
RESULT = run_route(SPEC)


{
  "route": "onnx800",
  "status": "BLOCKED_SETUP",
  "issues": [
    "INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:book"
  ],
  "specification": {
    "name": "onnx800",
    "mode": "rerank",
    "depth": 800
  }
}


<div dir="rtl" align="right">

### NB22E-C007 — مدل۴۰۰ + ترکیب رتبه

اجرای کامل همین مسیر، ذخیره، ارزیابی و گزارش؛ بدون وابستگی به خروجی مسیر دیگر.

</div>

In [7]:
# NB22E-C007 — Independent complete route: onnx400_fusion
SPEC = {'name': 'onnx400_fusion', 'mode': 'fusion', 'depth': 400}
RESULT = run_route(SPEC)


{
  "route": "onnx400_fusion",
  "status": "BLOCKED_SETUP",
  "issues": [
    "INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:book"
  ],
  "specification": {
    "name": "onnx400_fusion",
    "mode": "fusion",
    "depth": 400
  }
}


<div dir="rtl" align="right">

### NB22E-C008 — جدول تصمیم نهایی

مقایسهٔ جمعیت یکسان، حد پذیرش و تصمیم واقعی؛ اجرای بزرگ‌تر خودکار نیست.

</div>

In [8]:
# NB22E-C008 — Compare identical complete pilot cohorts; no full-run dispatch.
rows=[];baseline=LAB['results'].get('baseline_D',{});reference=baseline.get('metrics');eligible_rows=[]
for spec in ROUTES:
 name=spec['name'];r=LAB['results'].get(name,{'status':'NOT_RUN'});m=r.get('metrics')
 row={'route':name,'status':r['status'],'query_count':r.get('query_count',0),'metrics':m,
  'new_pairs':r.get('new_pairs',0),'seconds_incremental':r.get('route_wall_seconds'),
  'pool_ceiling_not_actual_top10':r.get('pool_ceiling')}
 if m is not None:
  row['acceptance']={k:{'threshold':v,'status':'PASS' if m[k]>=v else 'FAIL','mark':'✅' if m[k]>=v else ''} for k,v in CONFIG['thresholds'].items()}
  row['hit_absolute_threshold']=None
  row['eligible_for_larger_development_pilot']=name!='baseline_D' and reference is not None and eligible(m,reference)
  if row['eligible_for_larger_development_pilot']:eligible_rows.append(row)
 rows.append(row)
winner=sorted(eligible_rows,key=lambda r:(-r['metrics']['ndcg_at_10'],r['route']))[0]['route'] if eligible_rows else None
print('\n| مسیر | وضعیت | سؤال | Recall@10؛ حد 85% | قبول | nDCG@10؛ حد 75% | قبول | Hit@10؛ حد تعیین نشده | جفت تازه | ثانیهٔ افزایشی |')
print('|---|---|---:|---:|:---:|---:|:---:|---:|---:|---:|')
for row in rows:
 m=row['metrics'];a=row.get('acceptance',{})
 print('| '+row['route']+' | '+row['status']+' | '+str(row['query_count'])+' | '+
  (' | '.join([f"{m['recall_at_10']*100:.2f}%",a['recall_at_10']['mark'],f"{m['ndcg_at_10']*100:.2f}%",a['ndcg_at_10']['mark'],f"{m['hit_at_10']*100:.2f}%"]) if m else '— | | — | | —')+
  ' | '+str(row['new_pairs'])+' | '+(f"{row['seconds_incremental']:.1f}" if row['seconds_incremental'] is not None else '—')+' |')
summary={'schema_version':'NB22E_LAB_V1_FINAL','pilot_complete_for_all_routes':all(r['status']=='COMPLETE' for r in rows),
 'setup_ready':LAB['ready'],'issues':LAB['issues'],'routes':rows,'exploratory_winner':winner,
 'next_step':'REVIEW_PILOT_QUALITY_AND_TOTAL_COMPUTE_BEFORE_LARGER_DEVELOPMENT_RUN' if winner else 'KEEP_NB22D_AND_REVIEW_NEGATIVE_OR_HELD_ROUTES',
 'new_cache_miss_pairs':LAB['new_pairs'],'runtime_parity_check_pairs':LAB['parity_pairs'],
 'total_new_model_pair_evaluations':LAB['new_pairs']+LAB['parity_pairs'],'new_api_calls':0,'downloads':False,'test_evaluation':False,
 'release_acceptance_granted':False,'pilot_is_not_independent_acceptance_or_statistical_proof':True,
 'runtime_note':'Shared caches make later routes cheaper. Do not compare incremental route time as standalone speed. No full-cohort duration extrapolation is claimed from cache-hit timings.',
 'source_inputs':STATE['inputs'],'producer_code_sha256':NOTEBOOK_CODE_HASH}
if LAB['ready']:
 summary['plan']=LAB['plan'];summary['report']=save('A02.lab_comparison',dict(summary))
print(json.dumps(summary,ensure_ascii=False,indent=2))



| مسیر | وضعیت | سؤال | Recall@10؛ حد 85% | قبول | nDCG@10؛ حد 75% | قبول | Hit@10؛ حد تعیین نشده | جفت تازه | ثانیهٔ افزایشی |
|---|---|---:|---:|:---:|---:|:---:|---:|---:|---:|
| baseline_D | BLOCKED_SETUP | 0 | — | | — | | — | 0 | — |
| onnx200 | BLOCKED_SETUP | 0 | — | | — | | — | 0 | — |
| onnx400 | BLOCKED_SETUP | 0 | — | | — | | — | 0 | — |
| onnx800 | BLOCKED_SETUP | 0 | — | | — | | — | 0 | — |
| onnx400_fusion | BLOCKED_SETUP | 0 | — | | — | | — | 0 | — |
{
  "schema_version": "NB22E_LAB_V1_FINAL",
  "pilot_complete_for_all_routes": false,
  "setup_ready": false,
  "issues": [
    "INSUFFICIENT_DISTINCT_SOURCE_COMPONENTS:book"
  ],
  "routes": [
    {
      "route": "baseline_D",
      "status": "BLOCKED_SETUP",
      "query_count": 0,
      "metrics": null,
      "new_pairs": 0,
      "seconds_incremental": null,
      "pool_ceiling_not_actual_top10": null
    },
    {
      "route": "onnx200",
      "status": "BLOCKED_SETUP",
      "query_count": 0,
      "metrics": null,


<div dir="rtl" align="right">

## حدود آزمون نسخهٔ v1

۲۲ کنترل نرم‌افزاری با دادهٔ مصنوعی و پیش‌بینی‌گر آزمایشی پاس شده است: هویت و Schema، ورودی‌های پین‌شده، نمونهٔ مشترک، هر پنج مسیر، استقلال سل‌های مسیر، اشتراک کش، عدم محاسبهٔ دوبارهٔ جفت، سقف محاسبات، حذف مسیر ناقص از انتخاب، کش خراب، تغییر متن و ورودی ناقص. شبکه در آزمون‌ها مسدود بود.

این آزمون‌ها اجرای واقعی ONNX، زمان مک کاربر یا بهبود واقعی پروژه را اثبات نمی‌کنند؛ این‌ها پس از اجرای مدل و ورودی واقعی مشخص می‌شوند. در این نسخه مدل جایگزین تازه دانلود یا معرفی نشده و مقایسهٔ مدل‌های مختلف ادعا نمی‌شود.

</div>